# M4 · LangChain ve LangGraph
**Süre:** 20 dk canlı kodlama · **Demo çıktısı:** Araç kullanan, durumlu tek ajan

| | LangChain | LangGraph |
|---|---|---|
| Ne? | Yapı taşları: model, prompt, **araç (tool)**, retriever | **Durum makinesi:** düğüm (node), kenar (edge), durum (state) |
| Ne zaman? | Doğrusal zincirler | Döngü, karar, hafıza, insan onayı gereken akışlar |

```
  START ──▶ [asistan] ──(araç çağrısı var mı?)──evet──▶ [araçlar]
               ▲                 │                         │
               │                hayır                      │
               │                 ▼                         │
               │                END                        │
               └───────────────────────────────────────────┘
```
**Durumlu (stateful):** Checkpointer her adımda durumu `thread_id` ile saklar → ajan konuşmayı hatırlar.

In [ ]:
# ⚙️ Ortam hazırlığı: Colab'da ve yerel Jupyter'da çalışır
import os, sys, subprocess
REPO_URL = "https://github.com/volkanak/MBT-GenAI-Egitim.git"   # ← kendi GitHub reponuz
KOK = next((os.path.abspath(p) for p in [".", "..", "/content/MBT-GenAI-Egitim"]
            if os.path.exists(os.path.join(p, "ortak.py"))), None)
if KOK is None:  # Colab: repoyu klonla
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/MBT-GenAI-Egitim"], check=True)
    KOK = "/content/MBT-GenAI-Egitim"
try:
    import langgraph, chromadb, langchain_anthropic, sentence_transformers, langchain_mcp_adapters  # noqa
except ImportError:
    print("Paketler kuruluyor (1-3 dk)…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KOK}/requirements.txt"], check=True)
sys.path.insert(0, KOK); os.chdir(KOK)
from ortak import *
api_anahtari_yukle()
print("Hazır ✔  Model:", MODEL)

## 1️⃣ Model

In [ ]:
llm = sohbet_modeli()
print(llm.invoke("Tek cümlede: ajan (agent) nedir?").text)

## 2️⃣ Araçlar (tools)  ✍️ *canlı kodlama*
Docstring **modelin okuduğu kullanım kılavuzudur**. Ne zaman kullanılacağını net yazın.

In [ ]:
from langchain_core.tools import tool

# ✍️ TODO: politika_ara, onay_yetkilisi_bul ve hesapla araçlarını @tool ile tanımlayın
...

araclar = [politika_ara, onay_yetkilisi_bul, hesapla]
llm_araclarla = llm.bind_tools(araclar)

# Model aracı kendisi çalıştırmaz; sadece "şu aracı şu argümanla çağır" der:
print(llm_araclarla.invoke("1.200.000 TL'lik alımı kim onaylar?").tool_calls)

## 3️⃣ Grafı kur: düğümler, kenarlar, hafıza  ✍️ *canlı kodlama*

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

SISTEM = SystemMessage(
    "Sen Atlas Teknoloji'nin satın alma ve politika asistanısın. Kurallar için mutlaka politika_ara aracını kullan, "
    "hesaplamaları hesapla aracıyla yap, onay sorularında onay_yetkilisi_bul kullan. Cevapta kaynak [DOK-xx] göster."
)

# ✍️ TODO: asistan düğümü + ToolNode + koşullu kenar + InMemorySaver ile grafı derleyin
...

In [ ]:
from IPython.display import Image, display
try:
    display(Image(ajan.get_graph().draw_mermaid_png()))
except Exception:
    print(ajan.get_graph().draw_mermaid())

## 4️⃣ Çalıştır: ajanın adımlarını izleyelim

In [ ]:
def calistir(soru: str, thread_id: str):
    ayar = {"configurable": {"thread_id": thread_id}}
    for adim in ajan.stream({"messages": [("user", soru)]}, ayar, stream_mode="updates"):
        for dugum, guncelleme in adim.items():
            for m in guncelleme["messages"]:
                if getattr(m, "tool_calls", None):
                    for tc in m.tool_calls:
                        print(f"🔧 {tc['name']}({tc['args']})")
                elif m.type == "tool":
                    print(f"   ↳ {m.content[:120].replace(chr(10), ' ')}…")
                else:
                    print(f"\n🤖 {m.text}")

calistir("Delta Yazılım'a 2.000.000 TL'lik sözleşme için %40 avans vermek istiyoruz. "
         "Avans tutarı ne olur, politikaya uygun mu ve sözleşmeyi kim onaylamalı?", thread_id="t1")

## 5️⃣ Durumlu: aynı thread hatırlar, yeni thread hatırlamaz

In [ ]:
calistir("Peki avansı politikaya uygun en yüksek orana çekersek tutar ne olur?", thread_id="t1")

In [ ]:
calistir("Peki avansı politikaya uygun en yüksek orana çekersek tutar ne olur?", thread_id="t2")

In [ ]:
durum = ajan.get_state({"configurable": {"thread_id": "t1"}})
print("t1 thread'indeki mesaj sayısı:", len(durum.values["messages"]))

### ✅ Özet ve tartışma
- **Ajan = LLM + araçlar + döngü.** Hangi aracı ne zaman çağıracağına model karar verir.
- **Checkpointer** üretimde Postgres/Redis olur → kullanıcı ertesi gün kaldığı yerden devam eder.
- `interrupt()` ile **insan onayı** düğümü eklenebilir (örn. 500.000 TL üstü kararlar).
- Tek ajan çok fazla araç ve sorumluluk alınca zorlanır → **M5: uzman ajanlardan oluşan ekip**